In [2]:
import os
import pandas as pd
import geopandas as gpd
from shapely import wkt
import matplotlib

### Get census data

In [3]:
import mxcensus #Gonzalo's lib

# Full pipeline for one state (14 = Jalisco)
census = mxcensus.load_census(state=14)

# Extended-questionnaire microdata
personas = mxcensus.load_extended_personas(state=14)
viviendas = mxcensus.load_extended_viviendas(state=14)

# Geometries (Marco Geoestadístico) merged with census counts
mg_aur, mg_loc_ageb = mxcensus.load_mg_census(state=14)

# DENUE economic units — any release, harmonized to the latest schema by default
denue = mxcensus.load_denue(state=14)                      # latest release
denue_2010 = mxcensus.load_denue(state=14, release="201000")   # comparable to latest
raw = mxcensus.load_denue(state=14, release="201000", harmonize=False)  # raw schema

In [ ]:
folder = r"/Users/jeannettearjona/Documents/censo jalisco"
mg_aur.to_file(os.path.join(folder, 'mg_aur.shp')) #agebs rurales y urbanos

#mg_loc_ageb.to_file(os.path.join(folder, 'mg_loc_ageb.shp')) #sin agebs rurales (creo)
print('wont be working with mg_loc_ageb as it is a combination of:')
mg_loc_ageb.geometry.geom_type.value_counts()

wont be working with mg_loc_ageb as it is a combination of:


MultiPoint      17534
MultiPolygon     6504
Name: count, dtype: int64

In [5]:
# Estimacion de empleos por establecimiento
per_ocu_map = {
    '0 a 5 personas': 2.5,
    '6 a 10 personas': 8,
    '11 a 30 personas': 20.5,
    '31 a 50 personas': 40.5,
    '51 a 100 personas': 75.5,
    '101 a 250 personas': 175.5,
    '251 y más personas': 300
}
denue['empleos_est'] = denue['per_ocu'].map(per_ocu_map)

denue_ageb = (
    denue.groupby(['cve_ent','cve_mun', 'cve_loc', 'ageb'])
    .agg(
        establecimientos_tot = ('id', 'nunique'),
        empleos_tot = ('empleos_est', 'sum')
    )
    .reset_index()
)
denue_ageb

,cve_ent,cve_mun,cve_loc,ageb,establecimientos_tot,empleos_tot
0,14,001,0001,0067,93,261.5
1,14,001,0001,0071,302,1075.5
2,14,001,0001,0103,49,133.5
3,14,001,0001,0118,24,65.5
4,14,001,0001,0122,106,436.0
...,...,...,...,...,...,...
5339,14,125,0022,012A,1,75.5
5340,14,125,0057,012A,1,20.5
5341,14,125,0064,0115,1,20.5
5342,14,125,0075,012A,1,8.0


### AGEBS with census & denue data (POBTOT, Tot empleos, Tot establecimientos)

In [6]:
agebs_censo = mg_aur.copy()
agebs_censo.reset_index(inplace=True)
agebs_censo['ENTIDAD'] = agebs_censo['ENTIDAD'].astype(str)
agebs_censo['MUN'] = agebs_censo['MUN'].astype(str).str.zfill(3)
agebs_censo['LOC'] = agebs_censo['LOC'].astype(str).str.zfill(4)

agebs_censo = agebs_censo.merge(
    denue_ageb,
    left_on=['ENTIDAD', 'MUN', 'LOC', 'AGEB'],
    right_on=['cve_ent', 'cve_mun', 'cve_loc', 'ageb'],
    how='left'
)
agebs_censo.fillna({'establecimientos_tot':0, 'empleos_tot':0}, inplace=True)
agebs_censo


,ENTIDAD,MUN,LOC,AGEB,CVEGEO,geometry,ADMIN_TYPE,POBTOT,POBFEM,POBMAS,...,VPH_SINTIC,POBCOL,TOTCOL,PAFIL_PUB,cve_ent,cve_mun,cve_loc,ageb,establecimientos_tot,empleos_tot
0,14,001,0000,0014,140010014,"MULTIPOLYGON (((2410828.699 985273.726, 241092...",AGEB_RURAL,1044,499,523,...,5,0,0,648,NaN,NaN,NaN,NaN,0.0,0.0
1,14,001,0000,0029,140010029,"MULTIPOLYGON (((2413423.301 982641.06, 2413514...",AGEB_RURAL,677,322,331,...,0,0,0,434,NaN,NaN,NaN,NaN,0.0,0.0
2,14,001,0000,0048,140010048,"MULTIPOLYGON (((2397147 977105.501, 2397194 97...",AGEB_RURAL,2195,1073,1082,...,7,0,0,987,NaN,NaN,NaN,NaN,0.0,0.0
3,14,001,0000,0052,140010052,"MULTIPOLYGON (((2415459 973010.25, 2415496.3 9...",AGEB_RURAL,3332,1676,1618,...,6,6,1,2177,NaN,NaN,NaN,NaN,0.0,0.0
4,14,001,0001,0067,1400100010067,"MULTIPOLYGON (((2404841.304 974947.259, 240490...",AGEB_URBAN,1266,639,627,...,<NA>,0,0,<NA>,14,001,0001,0067,93.0,261.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5868,14,125,0001,0098,1412500010098,"MULTIPOLYGON (((2445010.666 970700.808, 244505...",AGEB_URBAN,1061,583,478,...,3,0,0,<NA>,14,125,0001,0098,98.0,292.0
5869,14,125,0001,0100,1412500010100,"MULTIPOLYGON (((2444637.848 970322.391, 244468...",AGEB_URBAN,1272,638,634,...,<NA>,15,1,<NA>,14,125,0001,0100,57.0,243.5
5870,14,125,0001,0134,1412500010134,"MULTIPOLYGON (((2445289.38 971559.069, 2445280...",AGEB_URBAN,131,64,67,...,0,0,0,79,14,125,0001,0134,10.0,25.0
5871,14,125,0001,0149,1412500010149,"MULTIPOLYGON (((2443647.613 971193.485, 244367...",AGEB_URBAN,5,<NA>,<NA>,...,<NA>,0,0,<NA>,14,125,0001,0149,3.0,13.0


In [7]:
agebs_censo.crs

<Projected CRS: {"$schema": "https://proj.org/schemas/v0.7/projjso ...>
Name: MEXICO_ITRF_2008_LCC
Axis Info [cartesian]:
- [east]: Easting (metre)
- [north]: Northing (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: unnamed
- method: Lambert Conic Conformal (2SP)
Datum: International Terrestrial Reference Frame 2008
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

### Get distance to CVE_MUN center for each AGEB

In [56]:
# Jalisco = 19
localidades19 = pd.read_excel(r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Insumos Analisis/Localidades19.xlsx', skiprows=3)
# cabeceras = LOC == 1
cabeceras19 = localidades19[localidades19['CVE_LOC']==1]
cabeceras19 = gpd.GeoDataFrame(
    cabeceras19,
    geometry= gpd.points_from_xy(
        cabeceras19['LON_DECIMAL'],
        cabeceras19['LAT_DECIMAL']
    ),
    crs="EPSG:4326"
)
cabeceras19 = cabeceras19.to_crs("EPSG:32613")
cabeceras19['CVE_MUN'] = cabeceras19['CVE_MUN'].astype(str).str.zfill(3)

# sacar centroide de cada AGEB (agebs_censo)
agebs_censo = agebs_censo.to_crs("EPSG:32613")
agebs_censo['centroide'] = agebs_censo.geometry.representative_point()

#merge on MUN
agebs = agebs_censo.copy()
agebs = agebs.merge(
    cabeceras19[['CVE_MUN', 'geometry']],
    left_on="MUN",
    right_on="CVE_MUN",
    how="left",
    suffixes=("","_cabecera")
)

#calc distance
agebs["dist_toCAB"] = agebs["centroide"].distance(agebs["geometry_cabecera"])
agebs["dist_toCAB_km"] = agebs["dist_toCAB"]/1000

agebs = agebs.to_crs("EPSG:4326")
agebs['cent_x'] = agebs['centroide'].x
agebs['cent_y'] = agebs['centroide'].y
agebs['cab_x'] = agebs['geometry_cabecera'].x
agebs['cab_y'] = agebs['geometry_cabecera'].y
agebs = agebs.drop(columns=['centroide', 'geometry_cabecera'])

In [58]:
agebs.to_file(os.path.join(folder, 'agebs_censo_14.shp')) #agebs rurales y urbanos

/var/folders/8s/00_wwq9j23b09mnd7gp105m00000gp/T/ipykernel_1052/3454881835.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  agebs.to_file(os.path.join(folder, 'agebs_censo_14.shp')) #agebs rurales y urbanos
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'PCDISC_LENG' to 'PCDISC_LEN'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'PCDISC_MOT2' to 'PCDISC_M_1'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'PCLIM_RE_CO' to 'PCLIM_RE_C'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'PAFIL_IPRIV' to 'PAFIL_IPRI'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field nam

### Read Visum Links & find ageb of their midpoint

In [61]:
visum_links = r"/Users/jeannettearjona/Downloads/OneDrive_1_6-10-2026/edges_from_visum.shp"
gdl_links = gpd.read_file(visum_links)

# reproject to crs metrico
gdl_links = gdl_links.to_crs("EPSG:32613")
midpoints = gdl_links.copy()
midpoints['geometry'] = gdl_links.geometry.interpolate(0.5, normalized=True)
midpoints = midpoints.to_crs("EPSG:4326")
gdl_links = gdl_links.to_crs("EPSG:4326")
midpoints.to_file(r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Visum Objects/Links/links_midpoint.shp')

In [63]:
print("midpoints CRS:", midpoints.crs)
print("agebs CRS:", agebs.crs)

print("midpoints bounds:", midpoints.total_bounds)
print("agebs bounds:", agebs.total_bounds)

midpoints CRS: EPSG:4326
agebs CRS: EPSG:4326
midpoints bounds: [-103.63597644   20.3412864  -102.87755842   20.97152396]
agebs bounds: [-105.69539424   18.92586906 -101.51053348   22.75024233]


### Spatial join of links midpoint to find in which AGEB falls

In [67]:
agebs_19 = agebs[["MUN",
    "LOC",
    "AGEB",
    "CVEGEO",
    "ADMIN_TYPE",
    "POBTOT",
    "establecimientos_tot",
    "empleos_tot",
    "dist_toCAB",
    "dist_toCAB_km",
    "geometry"]].copy()

midpoints = midpoints.to_crs(agebs_19.crs)

midpoints_in_ageb = gpd.sjoin(
    midpoints,
    agebs_19,
    how="left",
    predicate="within"
)
midpoints_in_ageb.to_file(os.path.join(folder, 'midpoints.shp')) #agebs rurales y urbanos

/var/folders/8s/00_wwq9j23b09mnd7gp105m00000gp/T/ipykernel_1052/446906977.py:21: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  midpoints_in_ageb.to_file(os.path.join(folder, 'midpoints.shp')) #agebs rurales y urbanos
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'index_right' to 'index_righ'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'establecimientos_tot' to 'establecim'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'empleos_tot' to 'empleos_to'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'dist_toCAB_km' to 'dist_toC_1'
  ogr_write(


### Merge Visum Links with midpoints census/denue data

In [5]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import numpy as np
from functools import lru_cache
from collections import defaultdict
from shapely.geometry import LineString
from shapely import wkt

#Open network .ver file (from local disk not onedrive)
import win32com.client

#Visum = com.Dispatch("Visum.Visum.250") #Add .250 for Visum 25 version
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Visum Projects\RedOSMNX_AMG_24 - Copy.ver')
C = win32com.client.constants

# === 1. Retrieve links ===
links = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "geometry": [i[1] for i in Visum.Net.Links.GetMultiAttValues("WKTPolyWGS84")],
    "FromNodeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "ToNodeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")],
    "TSysSet": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TSysSet")],
    "Length": [i[1] for i in Visum.Net.Links.GetMultiAttValues("Length")],
    "Cap_Transcad": [i[1] for i in Visum.Net.Links.GetMultiAttValues("CAPACIDAD_TRANSCAD")],
    "Vel_Transcad": [i[1] for i in Visum.Net.Links.GetMultiAttValues("VELOCIDAD_TRANSCAD")],
    "LimVel_Transcad": [i[1] for i in Visum.Net.Links.GetMultiAttValues("LIMVEL_TRANSCAD")],
})

In [10]:
midpoints_path = r"C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Visum Objects\Links\midpoints.shp"
midpoints = gpd.read_file(midpoints_path)
len(midpoints)

593968

In [9]:
midpoints

'C:\\Users\\AP03542515\\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\\Modelación Urbana - Red Vial Guadalajara\\Visum Objects\\Links\\midpoints.shp'